# Post-Training Threshold Synchronization

**Hypothesis:** Classification works best when output neurons spike synchronously. Neurons that spike late (after `t_objective`) produce weaker features under the `TargetRelative` decoder. We compute per-sample drift corrections to push late-wave neurons toward `t_objective`.

Model: `tobj=0.875, seed=1`

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

from applications.datasets import create_dataset
from applications.threshold_research.neuron_perturbation import (
    precompute_cumulative_potentials,
    spike_times_from_potentials,
    collect_input_times,
)
from applications.threshold_research.threshold_sync import (
    filter_to_first_waves,
    compute_sync_thresholds,
)
from spiking import load_model
from spiking.evaluation.decoding import TargetRelative
from spiking.evaluation.eval_classifier import evaluate_classifier
from spiking.evaluation.feature_extraction import extract_spike_times
from spiking.layers import SpikingSequential

MODEL_DIR = "logs/mnist/threshold_research/tobj_0.875/seed_1"
T_OBJECTIVE = 0.875
N_WAVES = 3

## Load model & data

In [ ]:
model = load_model(f"{MODEL_DIR}/model.pth")
layer = model.layers[0]
sub_model = SpikingSequential(*model.layers[:1]).cpu()

train_loader, test_loader = create_dataset("mnist")
spike_shape = train_loader.dataset.image_shape

print(f"Neurons: {layer.num_outputs}")
print(f"Spike shape: {spike_shape}")

# Extract spike times for train and test
spike_times_train, labels_train = extract_spike_times(sub_model, train_loader, spike_shape)
spike_times_test, labels_test = extract_spike_times(sub_model, test_loader, spike_shape)
print(f"Train spike times: {spike_times_train.shape}")
print(f"Test spike times: {spike_times_test.shape}")

# Precompute cumulative potentials (needed by iterative sync)
weights = layer.weights.detach()
train_input_times = collect_input_times(train_loader)
test_input_times = collect_input_times(test_loader)

train_cum, train_boundary = precompute_cumulative_potentials(train_input_times, weights)
test_cum, test_boundary = precompute_cumulative_potentials(test_input_times, weights)
print(f"Cumulative potentials: train {train_cum.shape}, test {test_cum.shape}")

## Baseline accuracy

In [ ]:
decoder = TargetRelative(t_target=T_OBJECTIVE)

X_train_baseline = decoder.decode(spike_times_train).numpy()
X_test_baseline = decoder.decode(spike_times_test).numpy()
y_train = labels_train.numpy()
y_test = labels_test.numpy()

train_metrics, test_metrics = evaluate_classifier(X_train_baseline, y_train, X_test_baseline, y_test)
print(f"Baseline train accuracy: {train_metrics['accuracy']:.4f}")
print(f"Baseline test accuracy:  {test_metrics['accuracy']:.4f}")

## Analyze late spike waves

In [ ]:
filtered_train = filter_to_first_waves(spike_times_train, n_waves=N_WAVES)

# Per-neuron: how often does it participate in the first N_WAVES waves?
finite_mask = torch.isfinite(filtered_train)
participation_rate = finite_mask.float().mean(dim=0)  # (N,)
print(f"Neurons participating in first {N_WAVES} waves (mean rate): {participation_rate.mean():.3f}")
print(f"Neurons that never appear in first waves: {(participation_rate == 0).sum().item()}")

# Per-neuron mean spike time (from first waves only)
neuron_means = []
for j in range(filtered_train.shape[1]):
    col = filtered_train[:, j]
    finite = col[torch.isfinite(col)]
    neuron_means.append(finite.mean().item() if len(finite) > 0 else float("nan"))
neuron_means = np.array(neuron_means)

valid = ~np.isnan(neuron_means)
print(f"\nPer-neuron mean spike time (first {N_WAVES} waves):")
print(f"  Mean: {neuron_means[valid].mean():.4f}")
print(f"  Std:  {neuron_means[valid].std():.4f}")
print(f"  Min:  {neuron_means[valid].min():.4f}")
print(f"  Max:  {neuron_means[valid].max():.4f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(neuron_means[valid], bins=30, edgecolor="black")
axes[0].axvline(T_OBJECTIVE, color="red", linestyle="--", label=f"t_objective={T_OBJECTIVE}")
axes[0].set_xlabel("Mean spike time")
axes[0].set_ylabel("Count")
axes[0].set_title(f"Per-neuron mean spike time (first {N_WAVES} waves)")
axes[0].legend()

axes[1].hist(participation_rate.numpy(), bins=30, edgecolor="black")
axes[1].set_xlabel("Participation rate")
axes[1].set_ylabel("Count")
axes[1].set_title(f"Neuron participation in first {N_WAVES} waves")
plt.tight_layout()
plt.show()

## Compute synchronized thresholds (drift-based)

In [ ]:
original_thresholds = layer.thresholds.detach().clone()

new_thresholds = compute_sync_thresholds(
    train_cum, train_boundary, spike_times_train, original_thresholds,
    target_time=T_OBJECTIVE, n_waves=N_WAVES,
)

ratio = new_thresholds / original_thresholds
changed = ratio != 1.0
print(f"Threshold change stats (changed neurons only, n={changed.sum().item()}):")
print(f"  Mean ratio (new/old): {ratio[changed].mean():.4f}")
print(f"  Std ratio:            {ratio[changed].std():.4f}")
print(f"  Min ratio:            {ratio[changed].min():.4f}")
print(f"  Max ratio:            {ratio[changed].max():.4f}")
print(f"  Unchanged neurons:    {(~changed).sum().item()}")

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(original_thresholds.numpy(), new_thresholds.numpy(), alpha=0.5, s=10)
lims = [
    min(original_thresholds.min().item(), new_thresholds.min().item()),
    max(original_thresholds.max().item(), new_thresholds.max().item()),
]
ax.plot(lims, lims, "r--", label="no change")
ax.set_xlabel("Original threshold")
ax.set_ylabel("Synchronized threshold")
ax.set_title("Threshold adjustment (drift-based)")
ax.legend()
plt.tight_layout()
plt.show()

## Recompute spike times with synchronized thresholds

In [ ]:
num_neurons = layer.num_outputs

def recompute_spike_times(cum_potentials, boundary_times, thresholds):
    """Recompute spike times for all neurons with given thresholds."""
    B = cum_potentials.shape[0]
    result = torch.full((B, num_neurons), float("inf"))
    for j in range(num_neurons):
        result[:, j] = spike_times_from_potentials(
            cum_potentials[:, j, :], boundary_times, thresholds[j].item(),
        )
    return result

spike_times_train_sync = recompute_spike_times(train_cum, train_boundary, new_thresholds)
spike_times_test_sync = recompute_spike_times(test_cum, test_boundary, new_thresholds)

# Compare distributions
finite_before = spike_times_train[torch.isfinite(spike_times_train)]
finite_after = spike_times_train_sync[torch.isfinite(spike_times_train_sync)]
print(f"Before sync — mean: {finite_before.mean():.4f}, std: {finite_before.std():.4f}")
print(f"After sync  — mean: {finite_after.mean():.4f}, std: {finite_after.std():.4f}")

## Evaluate synchronized model

In [ ]:
X_train_sync = decoder.decode(spike_times_train_sync).numpy()
X_test_sync = decoder.decode(spike_times_test_sync).numpy()

train_metrics_sync, test_metrics_sync = evaluate_classifier(
    X_train_sync, y_train, X_test_sync, y_test,
)

print("=" * 50)
print(f"{'Metric':<20} {'Baseline':>10} {'Synchronized':>14}")
print("=" * 50)
for metric in ["accuracy", "f1", "precision", "recall"]:
    base_val = test_metrics[metric]
    sync_val = test_metrics_sync[metric]
    diff = sync_val - base_val
    print(f"{metric:<20} {base_val:>10.4f} {sync_val:>10.4f}   ({diff:+.4f})")
print("=" * 50)

## Diagnostic plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Before/after spike time histogram overlay
ax = axes[0]
bins = np.linspace(0.5, 1.1, 40)
ax.hist(finite_before.numpy(), bins=bins, alpha=0.5, label="Before sync", edgecolor="black")
ax.hist(finite_after.numpy(), bins=bins, alpha=0.5, label="After sync", edgecolor="black")
ax.axvline(T_OBJECTIVE, color="red", linestyle="--", label=f"t_objective={T_OBJECTIVE}")
ax.set_xlabel("Spike time")
ax.set_ylabel("Count")
ax.set_title("Spike time distribution (train, all neurons)")
ax.legend()

# Per-neuron: mean spike time before vs after
ax = axes[1]
neuron_means_before = []
neuron_means_after = []
for j in range(num_neurons):
    col_before = spike_times_train[:, j]
    col_after = spike_times_train_sync[:, j]
    fb = col_before[torch.isfinite(col_before)]
    fa = col_after[torch.isfinite(col_after)]
    neuron_means_before.append(fb.mean().item() if len(fb) > 0 else float("nan"))
    neuron_means_after.append(fa.mean().item() if len(fa) > 0 else float("nan"))

neuron_means_before = np.array(neuron_means_before)
neuron_means_after = np.array(neuron_means_after)
valid = ~(np.isnan(neuron_means_before) | np.isnan(neuron_means_after))

ax.scatter(neuron_means_before[valid], neuron_means_after[valid], alpha=0.5, s=10)
lims = [
    min(neuron_means_before[valid].min(), neuron_means_after[valid].min()),
    max(neuron_means_before[valid].max(), neuron_means_after[valid].max()),
]
ax.plot(lims, lims, "r--", label="no change")
ax.axhline(T_OBJECTIVE, color="green", linestyle=":", alpha=0.7, label=f"target={T_OBJECTIVE}")
ax.set_xlabel("Mean spike time (before)")
ax.set_ylabel("Mean spike time (after)")
ax.set_title("Per-neuron mean spike time shift")
ax.legend()

plt.tight_layout()
plt.show()